# Build an Agent

**remark** - this is a tutorial building a simple agent using only LangChain. LangGraph is for building more advance agents and not covered here. 

**opinion** - This really is a great tutorial on the langchain site. It touches all important concepts in a very clear manner.

https://python.langchain.com/docs/tutorials/agents/#installation

In [24]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, tavily_api_key

#openai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key
os.environ['TAVILY_API_KEY'] = tavily_api_key

## End-to-end agent

The code snippet below represents a fully functional agent that uses an LLM to decide which tools to use. It is equipped with a generic search tool. It has conversational memory - meaning that it can be used as a multi-turn chatbot.

In the rest of the guide, we will walk through the individual components and what each part does - but if you want to just grab some code and get started, feel free to use this!

In [25]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

In [26]:
# create the agent

# This checkpoint saver stores checkpoints in memory using a defaultdict.
memory = MemorySaver()

# initiate a chat model --> new way
model = init_chat_model("openai:gpt-4o")

search = TavilySearch(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

In [27]:
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}

input_message = {
    "role": "user",
    "content": "Hi, I'm Sacha and I live in Voorschoten in the Netherlands.",
}

In [28]:
messages = {"messages": [input_message]}

for step in agent_executor.stream(messages, config, stream_mode="values"):
    # by using [-1] we only print the latest new message in the state messages
    step['messages'][-1].pretty_print()  

================================ Human Message =================================

Hi, I'm Sacha and I live in Voorschoten in the Netherlands.
================================== Ai Message ==================================

Hello Sacha! How can I assist you today?


In [37]:
input_message = {
    "role": "user",
    "content": "What's the weather where I live?",
}

for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather where I live?
================================== Ai Message ==================================

The current weather in Voorschoten, Netherlands, is cloudy with a temperature of 16.2°C (61.2°F). The wind is blowing from the north-northeast at 20.9 km/h (13.0 mph), and the humidity is at 48%. There is no precipitation at the moment.


In [31]:
state = agent_executor.get_state(config)

In [36]:
for msg in state.values['messages']:
    print(msg.content)

Hi, I'm Sacha and I live in Voorschoten in the Netherlands.
Hello Sacha! How can I assist you today?
What's the weather where I live?

{"query": "current weather Voorschoten Netherlands", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in Voorschoten, Netherlands", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'Voorschoten', 'region': 'South Holland', 'country': 'Netherlands', 'lat': 52.1275, 'lon': 4.4486, 'tz_id': 'Europe/Amsterdam', 'localtime_epoch': 1758541989, 'localtime': '2025-09-22 13:53'}, 'current': {'last_updated_epoch': 1758541500, 'last_updated': '2025-09-22 13:45', 'temp_c': 16.2, 'temp_f': 61.2, 'is_day': 1, 'condition': {'text': 'Cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/119.png', 'code': 1006}, 'wind_mph': 13.0, 'wind_kph': 20.9, 'wind_degree': 14, 'wind_dir': 'NNE', 'pressure_mb': 1024.0, 'pressure_in': 30.24, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 48, 'cloud': 0, 'feelslike_c

# Now step by step

## Define tools

We first need to create the tools we want to use. Our main tool of choice will be Tavily - a search engine. We can use the dedicated langchain-tavily integration package to easily use Tavily search engine as tool with LangChain.

In [44]:
from langchain_tavily import TavilySearch

search = TavilySearch(max_result=2)
search_results = search.invoke("What is the weather in Voorschote?")
print(search_results)

# If we want, we can create other tools
# Once the we have all the tools we want we put them in a list that we will reference
tools =[search]

{'query': 'What is the weather in Voorschote?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Voorschoten, Netherlands', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Voorschoten', 'region': 'South Holland', 'country': 'Netherlands', 'lat': 52.1275, 'lon': 4.4486, 'tz_id': 'Europe/Amsterdam', 'localtime_epoch': 1758548297, 'localtime': '2025-09-22 15:38'}, 'current': {'last_updated_epoch': 1758547800, 'last_updated': '2025-09-22 15:30', 'temp_c': 16.0, 'temp_f': 60.8, 'is_day': 1, 'condition': {'text': 'Patchy rain nearby', 'icon': '//cdn.weatherapi.com/weather/64x64/day/176.png', 'code': 1063}, 'wind_mph': 11.9, 'wind_kph': 19.1, 'wind_degree': 19, 'wind_dir': 'NNE', 'pressure_mb': 1024.0, 'pressure_in': 30.24, 'precip_mm': 0.02, 'precip_in': 0.0, 'humidity': 48, 'cloud': 0, 'feelslike_c': 16.0, 'feelslike_f': 60.8, 'windchill_c': 12.7, 'windchill_f': 54.8, 'heatindex_c': 14.1, 'heatindex_f': 57.4, 'dewpoint_c':

https://python.langchain.com/docs/how_to/custom_tools/

## Using Language Models

Next, let's learn how to use a language model to call tools. LangChain supports many different language models that you can use interchangably - select the one you want to use below!

In [45]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1", model_provider="openai")

In [47]:
query = "Hi!"
response = model.invoke([{"role": "user", "content": query}])

In [54]:
response.pretty_print()
# or
# response.content
# response.text

================================== Ai Message ==================================

Hello! How can I help you today? 😊


We can now see what it is like to enable this model to do tool calling. In order to enable that we use .bind_tools to give the language model knowledge of these tools

In [55]:
model_with_tools = model.bind_tools(tools)

We can now call the model. Let's first call it with a normal message, and see how it responds. We can look at both the content field as well as the tool_calls field.

In [56]:
query = "Hi!"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: Hello! How can I assist you today?

Tool calls: []


Now, let's try calling it with some input that would expect a tool to be called.



In [57]:
query = "Search for the weather in SF"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: 

Tool calls: [{'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco', 'search_depth': 'basic'}, 'id': 'call_sNfhCRfvaiUlKnlwSJA6b6WE', 'type': 'tool_call'}]


We can see that there's now no text content, but there is a tool call! It wants us to call the Tavily Search tool.

**This isn't calling that tool yet** - it's just telling us to. In order to actually call it, we'll want to create our agent.

## Create the agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using LangGraph to construct the agent. Currently, we are using a high level interface to construct the agent, but the nice thing about LangGraph is that this high-level interface is backed by a low-level, highly controllable API in case you want to modify the agent logic.

Now, we can initialize the agent with the LLM and the tools.

Note that we are passing in the `model`, not `model_with_tools`. That is because `create_react_agent` will call `.bind_tools` for us under the hood.

In [58]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

## Run the agent

We can now run the agent with a few queries! Note that for now, these are all stateless queries (it won't remember previous interactions). Note that the agent will return the final state at the end of the interaction (which includes any inputs, we will see later on how to get only the outputs).

First up, let's see how it responds when there's no need to call a tool:

In [59]:
input_message = {"role": "user", "content": "Hi!"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hi!
================================== Ai Message ==================================

Hello! How can I help you today?


In [60]:
input_message = {"role": "user", "content": "Search for the weather in SF"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Search for the weather in SF
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_Qvnqy2uXscj0W0KgBuQn1Iw6)
 Call ID: call_Qvnqy2uXscj0W0KgBuQn1Iw6
  Args:
    query: current weather in San Francisco
================================= Tool Message =================================
Name: tavily_search

{"query": "current weather in San Francisco", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in San Francisco", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1758548515, 'localtime': '2025-09-22 06:41'}, 'current': {'last_updated_epoch': 1758547800, 'last_updated': '2025-09-22 06:30', 'temp_c': 13.9, 'temp_f': 57.0, 'is

## Streaming Messages

We've seen how the agent can be called with .invoke to get a final response. If the agent executes multiple steps, this may take a while. To show intermediate progress, we can stream back messages as they occur.

In [61]:
for step in agent_executor.stream({"messages": [input_message]}, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Search for the weather in SF
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_kur3vBHxu8X6UGWoQnNuZt8D)
 Call ID: call_kur3vBHxu8X6UGWoQnNuZt8D
  Args:
    query: current weather in San Francisco
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "current weather in San Francisco", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "Weather in San Francisco", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1758548515, 'localtime': '2025-09-22 06:41'}, 'current': {'last_updated_epoch': 1758547800, 'last_updated': '2025-09-22 06:30', 'temp_c': 1

In [62]:
for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "agent" and (text := step.text()):
        print(text, end="|")

The| current| weather| in| San| Francisco| is| clear| with| a| temperature| of| about| |13|.|9|°C| (|57|°F|).| The| humidity| is| high| at| |96|%,| and| there| is| a| light| S|SW| wind| at| around| |3|.|8| mph| (|6|.|1| k|ph|).| The| weather| conditions| are| calm| and| visibility| is| good|.

|For| more| details|,| you| can| check| Weather|API|'s| San| Francisco| page|.|

## Adding in memory

As mentioned earlier, this agent is stateless. This means it does not remember previous interactions. To give it memory we need to pass in a checkpointer. When passing in a checkpointer, we also have to pass in a `thread_id` when invoking the agent (so it knows which thread/conversation to resume from).

In [65]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_executor = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "abc123"}}

for step in agent_executor.stream(
    {"messages": [("user", "Hi, I'm Bob!")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hi, I'm Bob!
================================== Ai Message ==================================

Hi Bob! How can I help you today?


In [66]:
for step in agent_executor.stream(
    {"messages": [("user", "What is my name?")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Bob! How can I assist you further?


If you want to start a new conversation, all you have to do is change the thread_id used

In [67]:
config = {"configurable": {"thread_id": "xyz123"}}

for step in agent_executor.stream(
    {"messages": [("user", "What is my name?")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

You haven’t provided your name yet. If you’d like, you can tell me your name, and I’ll remember it for this conversation!
